## **GITHUB repository**
Link: https://github.com/HABOZ1/Deliverable_2.git

In [21]:
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np


## **Task 1**

In [22]:
def file_reader(file_path: Path) -> tuple[str, str]:
    """
    Read a file and return its content along with the encoding used.

    Returns:
        A tuple containing the file content (str) and the encoding used (str).
    """

    try:
        # Attempt to read the file using UTF-8 encoding.
        with file_path.open("r", encoding="utf-8") as f:
            file_text_content = f.read()
            encoding_used = "utf-8"
        return (file_text_content, encoding_used)

    except UnicodeDecodeError:
        # If UTF-8 fails, try Latin-1.
        with file_path.open("r", encoding="latin-1") as f:
            file_text_content = f.read()
            encoding_used = "latin-1"
        return (file_text_content, encoding_used)


def log_parser(log_file_path: Path) -> list[dict[str, str | float]]:
    """
    Parse a log file and extract valid transactions.

    Returns:
        A list of dictionaries, each representing a valid transaction with keys:
        - timestamp (str)
        - register_id (str)
        - item_id (str)
        - price (float)
        - currency (str)
        - status (str)
    """
    # Read the file and retrieve only its content.
    file_content, _ = file_reader(log_file_path)

    # Create an empty list to store the extracted transactions.
    valid_transactions = []

    # Process the file one line at a time.
    for line in file_content.splitlines():

        # Skip printer warnings and debug messages that do not contain transactions.
        if (line.startswith("---") or line.startswith("===") or line.startswith("[DEBUG]")):
            continue

        else:
            # Extract the timestamp from inside the square brackets.
            timestamp = str(
                re.search(r"\[(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\]", line).group(1)
            )

            # Extract the remaining five transaction fields.
            register_id = str(re.search(r"POS_(\d+)", line).group(0))
            item_id = str(re.search(r"item=(\w+)", line).group(1))
            price = float(re.search(r"price=(\d+\.\d+)", line).group(1))
            currency = str(re.search(r"price=\d+\.\d+\s+(\w+)", line).group(1))
            status = str(re.search(r"\((SUCCESS|REFUND|CANCELLED)\)", line).group(1))

            # Store the extracted fields as a dictionary in the transaction list.
            valid_transactions.append(
                {
                    "timestamp": timestamp,
                    "register_id": register_id,
                    "item_id": item_id,
                    "price": price,
                    "currency": currency,
                    "status": status,
                }
            )

    # Return the list of all extracted transactions
    return valid_transactions


The file_reader() function reads a file using UTF-8 encoding, falling back to Latin-1 if decoding fails.

The log_parser() function processes the file line by line, skipping irrelevant lines and using regular expressions (the re module) to extract transaction data. Each transaction is stored as a dictionary containing the timestamp, register ID, item ID, price, currency and status. All transactions are collected in a list and returned.

The parser assumes that transaction lines follow the expected format, meaning malformed lines could cause errors.

## **Task 2**

In [23]:
def find_log_files(folder: Path) -> list[Path]:
    """
    Find all .log files in the specified directory and its subdirectories.

    Returns:
        A list of Path objects representing the found .log files.
    """
    log_files = []

    for path in folder.iterdir():
        if path.is_file() and path.suffix == ".log":
            log_files.append(path)

        elif path.is_dir():
            log_files.extend(find_log_files(path))

    return log_files


def find_active_registers(data: Path | dict | list) -> set[str]:
    """
    Find all active registers in a given JSON data structure.

    Returns:
        A set of strings representing the store names and IDs of active registers, e.g. "library_cafe/POS_01".
    """
    registers = set()

    # Check if the input is a Path object, and if so, read the JSON data from the file
    if isinstance(data, Path):
        with open(data, "r") as file:
            data = json.load(file)

    # Check if the input is a dictionary, and if so, look for "name", "id" and "active" keys
    if isinstance(data, dict):
        if "name" in data and "registers" in data:
            store_name = data["name"].lower().replace(" ", "_")

            for register in data["registers"]:
                if register["active"]:
                    registers.add(f"{store_name}/{register['id']}")

        for value in data.values():
            registers.update(find_active_registers(value))

    # Check if the input is a list, and if so, check each item in the list for a dictionary for "active" and "id" keys
    elif isinstance(data, list):
        for item in data:
            registers.update(find_active_registers(item))

    return registers


The find_log_files() function searches recursively through a directory structure using pathlib.Path. It checks each file and subdirectory, adding all files with the .log extension to a list. If a subdirectory is found, the function calls itself recursively to continue the search.

The find_active_registers() function recursively traverses the nested JSON structure containing store metadata. It handles both dictionaries and lists, searching for active registers. For each active register, the function stores a string containing the store name and register ID (for example, "library_cafe/POS_01") in a set. The recursive approach allows the function to process JSON files with arbitrary levels of nesting.

## **Task 3**

In [ ]:
def remove_price_outliers(log_df: pd.DataFrame) -> np.ndarray:
    """
    Remove outliers from price data in a DataFrame.

    Returns:
        A NumPy array of cleaned price data with outliers replaced by the median of non-outlier prices.
    """

    # Extract the price data from the input DataFrame, convert it to a 1D NumPy float array.
    price_data = log_df["price"].to_numpy(dtype=float).copy()

    # Calculate mean and standard deviation from the input price data.
    mean = np.mean(price_data)
    std_dev = np.std(price_data)

    # Detects outliers where a value deviates from the mean by more than 3 standard deviations. 
    outliers = np.abs(price_data - mean) > 3 * std_dev

    # Calculates the median from the valid price data. 
    median = np.median(price_data[~outliers])

    # Replaces the outliers with the median.
    price_data[outliers] = median

    # Returns the cleaned price data. 
    return price_data


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Computes the element-wise refund ratio using NumPy array operations.

    Returns:
        A Numpy Array with the computed refund ratios.
    """

    # Computes the refund-ratio with an added small constant to prevent zero-division errors.
    refund_ratio = np.array(refund_count / (sales_count + 10**-6))

    # Return the calculated refund ratios using a NumPy array.
    return refund_ratio


The remove_price_outliers() function identifies prices that deviate more than three standard deviations from the mean and replaces them with the median of the remaining prices.

The compute_refund_ratio() function calculates the refund ratio by dividing the number of refunds by the number of sales. A small constant is added to the denominator to avoid division by zero.

## **Task 4**

In [ ]:
def create_dataframe_from_logs(
    transactions: list[dict[str, str | float]]) -> pd.DataFrame:
    """
    Create a DataFrame from a list of transaction logs, and set the "timestamp" column as the index.

    Returns:
        A pandas DataFrame containing the transaction data, and the "timestamp" column is converted to datetime and set as the index.
    """

    # Create a DataFrame from the list of transactions. 
    df = pd.DataFrame(transactions)

    # Convert the "timestamp" column to datetime and set it as the index. 
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp").sort_index()

    return df


def group_by_item_and_hour(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group the DataFrame by "item_id", resample by hour and summing the "price" column.
    """

    return df.groupby("item_id").resample("1h")["price"].sum()


def create_revenue_pivot_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a pivot table showing total revenue for each item per hour.
    """

    pivot_table = pd.pivot_table(
        df,
        values="price",
        index="item_id",
        columns="status",
        aggfunc="sum",  # aggregate function to sum the prices for each item and status. 
        fill_value=0,  # fill_value to replace NaN values with 0. 
    )

    return pivot_table


def calculate_refund_ratio_per_item(df: pd.DataFrame) -> dict[str, float]:
    """
    Calculate the refund ratio for each item in the DataFrame.

    Returns:
        A dictionary where keys are item IDs and values are the corresponding refund ratios rounded to two decimals.
    """

    refund_ratios = {}

    # Loop through each unique item_id in the DataFrame. 
    for item in df["item_id"].unique():
        # Filter the DataFrame for the current item_id. 
        item_df = df[df["item_id"] == item]

        # Calculate the number of successful sales and refunds for the current item.
        sales_count = len(item_df[item_df["status"] == "SUCCESS"])
        refund_count = len(item_df[item_df["status"] == "REFUND"])

        # Compute the refund ratio and round it to two decimals.
        refund_ratios[item] = round(float(compute_refund_ratio(sales_count, refund_count)), 2)

    return refund_ratios


The create_dataframe_from_logs() function converts the list of transaction dictionaries into a Pandas DataFrame. The timestamp column is converted to a datetime format, set as the DataFrame index, and the DataFrame is sorted chronologically by timestamp.

The group_by_item_and_hour() function groups transactions by item type and resamples them into 1-hour time windows. For each item and time interval, the total transaction price is calculated using the sum aggregation function.

The create_revenue_pivot_table() function creates a pivot table showing the total transaction revenue for each item across the different transaction statuses (SUCCESS, REFUND, and CANCELLED). Missing values are replaced with 0 to make the table easier to read and analyse.

The calculate_refund_ratio_per_item() function calculates the refund ratio for each item type. The DataFrame is filtered by item, and the number of successful sales and refunds is counted. These values are then passed to the compute_refund_ratio() function from Task 3, and the results are stored in a dictionary.

## **Task 5**

In [26]:
# Find all log files and identify active registers.
log_file_paths = find_log_files(Path("../Deliverable_2/data/"))
active_registers = find_active_registers(Path("../Deliverable_2/data/store_config.json"))

# Create a list to store the DataFrames from active registers.
dataframes = []

# Process each log file and check whether its register is active.
for log_path in log_file_paths:

    # Combine the store name and register ID to identify the register uniquely.
    register_key = f"{log_path.parent.parent.name}/{log_path.parent.name}"

    if register_key in active_registers:
        # Parse the log file and convert its transactions into a DataFrame.
        df = create_dataframe_from_logs(log_parser(log_path))

        # Remove price outliers and correct prices recorded in NOK.
        df["price"] = remove_price_outliers(df)
        df.loc[df["currency"] == "NOK", "price"] *= 10.0

        # Add the store name and save the DataFrame for later combination.
        df["store"] = log_path.parent.parent.name
        dataframes.append(df)

# Combine all processed DataFrames into one and save the cleaned transactions.
combined_df = pd.concat(dataframes)
combined_df.to_csv("cleaned_transactions.csv", float_format="%.2f")

# Calculate hourly revenue per item and save the results as JSON.
hourly_revenue = group_by_item_and_hour(combined_df)
hourly_revenue.to_json("hourly_revenue.json", orient="table", indent=4)

# Create and save a pivot table of revenue grouped by item and transaction status.
pivot_table = create_revenue_pivot_table(combined_df)
pivot_table.to_csv("status_pivot_table.csv", float_format="%.2f")

# Calculate refund ratios per item and save them as a CSV file.
refund_ratio = calculate_refund_ratio_per_item(combined_df)
pd.DataFrame(refund_ratio.items(), columns=["item_id", "refund_ratio"]).to_csv("item_refund_ratios.csv", index=False)


The functions from the previous tasks are combined into one workflow. The program finds log files, filters for active registers, parses and cleans the transaction data, and corrects NOK prices. The processed data is combined into a single DataFrame, which is used to generate and save reports for hourly revenue, transaction status, and refund ratios.